# NannyML — End-to-End Model Monitoring Example

This notebook demonstrates the core features of [NannyML](https://nannyml.com), an open-source Python library for **post-deployment ML model monitoring**.

We cover:
1. Generating a synthetic loan-default dataset (binary classification)
2. Simulating production data with injected drift
3. **CBPE** — Confidence-Based Performance Estimation (no labels needed)
4. **Univariate Drift Detection** — per-feature statistical tests
5. **Multivariate Drift Detection** — PCA reconstruction error
6. **Ranking** — which features drifted most
7. **DLE** — Direct Loss Estimation for a regression model
8. **Period-based chunking** — weekly monitoring windows

> **Note:** The notebook is fully self-contained — no external files required.
> It also exports `reference_data.csv` and `analysis_data.csv` for reuse.


## 0. Install Dependencies

Run this once in your environment:

```bash
pip install nannyml scikit-learn pandas numpy matplotlib
```

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import nannyml as nml

print(f'NannyML version : {nml.__version__}')
print(f'Pandas  version : {pd.__version__}')
print(f'NumPy   version : {np.__version__}')

## 1. Generate Synthetic Loan-Default Dataset

We simulate a **loan default prediction** scenario:
- **Reference period** — stable data used to calibrate NannyML monitors
- **Analysis period** — later production data with injected covariate drift
  (lower incomes, worse credit scores, more unemployment)

Features: `age`, `income`, `loan_amount`, `credit_score`, `employment_status`, `loan_purpose`

In [ ]:
EMPLOYMENT_CATS = ['employed', 'self-employed', 'unemployed']
PURPOSE_CATS    = ['home', 'car', 'education', 'personal']

def make_dataset(n: int, drift: bool = False, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    age          = rng.integers(22, 65, size=n).astype(float)
    income       = rng.normal(55_000, 15_000, n).clip(10_000, 200_000)
    loan_amount  = rng.normal(12_000,  5_000, n).clip(1_000,  50_000)
    credit_score = rng.integers(300, 850, size=n).astype(float)
    emp          = rng.choice(EMPLOYMENT_CATS, n, p=[0.70, 0.20, 0.10])
    purpose      = rng.choice(PURPOSE_CATS, n)

    if drift:
        income       = (income * 0.75).clip(10_000, 200_000)
        credit_score = (credit_score - 80).clip(300, 850)
        emp          = rng.choice(EMPLOYMENT_CATS, n, p=[0.50, 0.20, 0.30])

    log_odds = (
        -3.0
        + 0.02  * (loan_amount / 1_000)
        - 0.004 * (credit_score - 500)
        - 0.01  * (income / 1_000)
        + 0.5   * (emp == 'unemployed')
        + rng.normal(0, 0.3, n)
    )
    prob_default = 1 / (1 + np.exp(-log_odds))
    y_true       = (rng.random(n) < prob_default).astype(int)
    y_pred_proba = (prob_default + rng.normal(0, 0.05, n)).clip(0.01, 0.99)
    y_pred       = (y_pred_proba >= 0.5).astype(int)

    return pd.DataFrame({
        'age':               age,
        'income':            income,
        'loan_amount':       loan_amount,
        'credit_score':      credit_score,
        'employment_status': emp,
        'loan_purpose':      purpose,
        'y_pred_proba':      y_pred_proba,
        'y_pred':            y_pred,
        'y_true':            y_true,
    })

reference        = make_dataset(5_000, drift=False, seed=42)
analysis_full    = make_dataset(5_000, drift=True,  seed=99)
analysis_no_gt   = analysis_full.drop(columns=['y_true'])   # simulates no ground-truth in prod

print('Reference shape :', reference.shape)
print('Analysis  shape :', analysis_full.shape)
reference.head()

## 2. Exploratory Distribution Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, feat in zip(axes.flatten(), ['age', 'income', 'loan_amount', 'credit_score']):
    ax.hist(reference[feat],     bins=40, alpha=0.6, label='Reference', color='steelblue', density=True)
    ax.hist(analysis_full[feat], bins=40, alpha=0.6, label='Analysis',  color='tomato',    density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle('Reference vs Analysis Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Export to CSV

In [ ]:
reference.to_csv('reference_data.csv', index=False)
analysis_full.to_csv('analysis_data.csv', index=False)
print('Saved reference_data.csv and analysis_data.csv')

## 4. CBPE — Confidence-Based Performance Estimation

Estimates **ROC-AUC, F1, Precision, Recall** from predicted probabilities alone —
no ground-truth labels needed in production.

In [ ]:
cbpe = nml.CBPE(
    y_pred_proba='y_pred_proba',
    y_pred='y_pred',
    y_true='y_true',
    problem_type='classification_binary',
    chunk_size=500,
    metrics=['roc_auc', 'f1', 'precision', 'recall'],
)
cbpe.fit(reference)
cbpe_results = cbpe.estimate(analysis_no_gt)

print('CBPE result columns:', cbpe_results.to_df().columns.tolist()[:8])
cbpe_results.to_df().head()

In [ ]:
cbpe_results.plot(kind='performance', metric='roc_auc').show()
cbpe_results.plot(kind='performance', metric='f1').show()

## 5. Univariate Drift Detection

Tests each feature independently:
- **KS test** (Kolmogorov-Smirnov) for continuous features
- **Chi-squared test** for categorical features

In [ ]:
FEATURE_COLS = ['age', 'income', 'loan_amount', 'credit_score',
                'employment_status', 'loan_purpose']

uni_calc = nml.UnivariateDriftCalculator(
    column_names=FEATURE_COLS,
    chunk_size=500,
)
uni_calc.fit(reference)
uni_results = uni_calc.calculate(analysis_no_gt)

print('Univariate drift results (first 3 rows):')
uni_results.to_df().head(3)

In [ ]:
for col in ['income', 'credit_score', 'employment_status']:
    uni_results.plot(kind='drift', column_name=col).show()

## 6. Multivariate Drift Detection

Uses **PCA reconstruction error** to catch joint drift not visible in individual features.

In [ ]:
multi_calc = nml.DataReconstructionDriftCalculator(
    column_names=['age', 'income', 'loan_amount', 'credit_score'],
    chunk_size=500,
)
multi_calc.fit(reference)
multi_results = multi_calc.calculate(analysis_no_gt)

print('Multivariate results (first 5 chunks):')
print(multi_results.to_df().head())

multi_results.plot(kind='drift').show()

## 7. Ranking — Which Features Drifted Most?

Ranks features by alert count to guide root-cause analysis.

In [ ]:
ranker = nml.AlertCountRanker()
ranked = ranker.rank(uni_results, only_drifting=False)
print('Feature ranking by drift alert count:')
print(ranked.to_string(index=False))

## 8. DLE — Direct Loss Estimation (Regression)

For **regression models**, DLE estimates RMSE and MAE without ground truth.
We use a **loan interest rate prediction** scenario to demonstrate.

In [ ]:
def make_regression_dataset(n: int, drift: bool = False, seed: int = 10) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    credit_score = rng.integers(300, 850, n).astype(float)
    loan_amount  = rng.normal(12_000, 5_000, n).clip(1_000, 50_000)
    income       = rng.normal(55_000, 15_000, n).clip(10_000, 200_000)
    if drift:
        credit_score = (credit_score - 80).clip(300, 850)
    y_true = (
        15.0 - 0.01 * (credit_score - 300)
             + 0.05 * (loan_amount / 1_000)
             - 0.02 * (income / 1_000)
             + rng.normal(0, 0.5, n)
    ).clip(3.0, 25.0)
    y_pred = (y_true + rng.normal(0, 0.8, n)).clip(3.0, 25.0)
    return pd.DataFrame({'credit_score': credit_score, 'loan_amount': loan_amount,
                         'income': income, 'y_pred': y_pred, 'y_true': y_true})

ref_reg       = make_regression_dataset(5_000, drift=False, seed=10)
anal_reg_full = make_regression_dataset(5_000, drift=True,  seed=77)
anal_reg_no_gt = anal_reg_full.drop(columns=['y_true'])
print('Regression reference shape:', ref_reg.shape)
ref_reg.head()

In [ ]:
dle = nml.DLE(
    feature_column_names=['credit_score', 'loan_amount', 'income'],
    y_pred='y_pred',
    y_true='y_true',
    metrics=['rmse', 'mae'],
    chunk_size=500,
)
dle.fit(ref_reg)
dle_results = dle.estimate(anal_reg_no_gt)

print('DLE Results (first 5 chunks):')
print(dle_results.to_df().head())

dle_results.plot(kind='performance', metric='rmse').show()
dle_results.plot(kind='performance', metric='mae').show()

## 9. Period-Based Chunking (Weekly)

| Strategy | Parameter | Description |
|---|---|---|
| Size-based | `chunk_size=500` | Fixed rows per chunk |
| Count-based | `chunk_number=10` | Fixed number of equal chunks |
| Period-based | `chunk_period='W'` | Time-window chunks (requires timestamp column) |

In [ ]:
# Add synthetic timestamps
ref_ts   = reference.copy()
anal_ts  = analysis_no_gt.copy()
ref_ts['timestamp']  = pd.date_range('2023-01-01', periods=len(ref_ts),  freq='H')
anal_ts['timestamp'] = pd.date_range('2023-04-01', periods=len(anal_ts), freq='H')

cbpe_weekly = nml.CBPE(
    y_pred_proba='y_pred_proba',
    y_pred='y_pred',
    y_true='y_true',
    problem_type='classification_binary',
    timestamp_column_name='timestamp',
    chunk_period='W',
    metrics=['roc_auc'],
)
cbpe_weekly.fit(ref_ts)
results_weekly = cbpe_weekly.estimate(anal_ts)

print('Weekly chunks:')
print(results_weekly.to_df()[['chunk_start_date', 'chunk_end_date']].head(8))

results_weekly.plot(kind='performance', metric='roc_auc').show()

## 10. Summary & Best Practices

| Task | NannyML Component | Labels Required? |
|---|---|---|
| Classification performance monitoring | `CBPE` | No |
| Regression performance monitoring | `DLE` | No |
| Per-feature drift detection | `UnivariateDriftCalculator` | No |
| Joint / multivariate drift | `DataReconstructionDriftCalculator` | No |
| Root-cause ranking | `AlertCountRanker` | No |

### Key Best Practices

1. **Fit on reference data only** — use your stable post-training production period.
2. **Calibrate model probabilities** before using CBPE for accurate estimation.
3. **Combine univariate + multivariate drift** — they catch complementary patterns.
4. **Choose chunk strategy wisely** — period-based is most interpretable in production.
5. **Tune alert thresholds** to your business tolerance for false positives.
6. **Persist your fitted calculators** with `pickle` so you don't need to re-fit each run:

```python
import pickle

# Save
with open('cbpe.pkl', 'wb') as f:
    pickle.dump(cbpe, f)

# Load & run on new data
with open('cbpe.pkl', 'rb') as f:
    cbpe_loaded = pickle.load(f)

new_results = cbpe_loaded.estimate(new_production_data)
```

7. **Integrate into your MLOps pipeline** — run NannyML on a schedule and push alerts to Slack, PagerDuty, or Grafana.
